In [1]:
import pandas as pd
import json
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix

df = pd.read_csv("../data/processed/training_data_with_history.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

with open("../data/processed/selected_features.json", "r") as file:
    raw_features = json.load(file)

history_features = ["smart_5_raw_change_1d", "smart_5_raw_change_7d", "smart_5_raw_change_30d", "smart_5_raw_rolling_mean_7d", "smart_5_raw_rolling_std_7d", "smart_5_raw_rolling_mean_30d"]

features = raw_features + history_features

train_end = pd.Timestamp("2026-02-09")
val_end = pd.Timestamp("2026-02-19")

X = df[features]
y = df["failure_within_30_days"]
train_mask = df["date"] <= train_end
val_mask = (df["date"] > train_end) & (df["date"] <= val_end)

X_train = X.loc[train_mask]
X_val = X.loc[val_mask]

y_train = y.loc[train_mask]
y_val = y.loc[val_mask]

pipeline = Pipeline([("imputer", SimpleImputer(strategy="median", add_indicator=True)), ("model", RandomForestClassifier(random_state=44))])

pipeline.fit(X_train, y_train)

failure_probabilities = pipeline.predict_proba(X_val)[:, 1]

predictions = (failure_probabilities >= 0.3).astype(int)

In [ ]:
val_results = df.loc[val_mask].copy()

val_results["failure_probability"] = failure_probabilities
val_results["prediction"] = predictions

false_negatives = val_results[(val_results["failure_within_30_days"] == 1) & (val_results["prediction"] == 0)]
false_positives = val_results[(val_results["failure_within_30_days"] == 0) & (val_results["prediction"] == 1)]

print("False negatives:", len(false_negatives))
print("False positives:", len(false_positives))

print(false_negatives[["serial_number", "date", "failure_probability", "smart_5_raw", "smart_5_raw_change_7d", "smart_5_raw_change_30d"]].head())


print(false_positives[["serial_number", "date", "failure_probability", "smart_5_raw", "smart_5_raw_change_7d", "smart_5_raw_change_30d"]].head())

False negatives: 643
False positives: 164
      serial_number       date  failure_probability  smart_5_raw  \
69565  88P0A0JRF97G 2026-02-10                 0.01          0.0   
69589  8190A0E5FVKG 2026-02-10                 0.08         46.0   
69631  6280A0V8FVKG 2026-02-10                 0.00          0.0   
69726  88Q0A0BCF97G 2026-02-10                 0.03          0.0   
69919  6250A00XFVKG 2026-02-10                 0.01          0.0   

       smart_5_raw_change_7d  smart_5_raw_change_30d  
69565                    0.0                     NaN  
69589                   19.0                    31.0  
69631                    0.0                     0.0  
69726                    0.0                     0.0  
69919                    0.0                     0.0  
          serial_number       date  failure_probability  smart_5_raw  \
69732  1a1cf23811210010 2026-02-10             0.389424          NaN   
69814          1QHJJKAX 2026-02-10             0.350000          0.0   
699

In [ ]:
from sklearn.inspection import permutation_importance

# Finds which features the model depends on when predicting the drive failures
importance_result = permutation_importance(pipeline, X_val, y_val, scoring="f1", n_repeats=5, random_state=44)

importance_df = pd.DataFrame({"Feature": features, "Importance": importance_result.importances_mean})
importance_df = importance_df.sort_values("Importance", ascending=False)


print(importance_df)

                         Feature  Importance
32  smart_5_raw_rolling_mean_30d    0.109606
29        smart_5_raw_change_30d    0.085660
30   smart_5_raw_rolling_mean_7d    0.043715
11                   smart_9_raw    0.038237
18                 smart_193_raw    0.032907


### Feature Importance

The permutation importance showed that the most important features included the 30-day rolling average, 30-day change and the 7-day rolling average of SMART 5. These history features were more important than the current SMART 5 value by itself and this supports the result from earlier that adding information about how SMART values change over time improved the model.